In [ ]:
import lightgbm as lgb
from sklearn.grid_search import GridSearchCV
import time
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [ ]:
import pandas as pd
data_tree_train = pd.read_csv('data_tree_train_final.csv',encoding='gbk')
data_tree_test = pd.read_csv('data_tree_test_final.csv',encoding='gbk')

data_tree_train = data_tree_train.fillna(0)
data_tree_test = data_tree_test.fillna(0)

data_tree_train_1 = data_tree_train[data_tree_train['label']==1]
data_tree_train_0 = data_tree_train[data_tree_train['label']==0]

train0,test0 = train_test_split(data_tree_train_0, test_size=0.1)
train1,test1 = train_test_split(data_tree_train_1, test_size=0.1)

test = pd.concat([test0,test1],ignore_index=True)

In [ ]:
train1Plus = train1.sample(13000,replace=True)
All = pd.concat([train0,train1Plus],ignore_index=True)

In [ ]:
All.label.value_counts()

In [ ]:
train1Plus = data_tree_train_1.sample(14000,replace=True)
train = pd.concat([data_tree_train_0,train1Plus],ignore_index=True)
train.label.value_counts()

In [ ]:
pd.set_option('display.max_colwidth',100)
pd.set_option('display.max_columns', 1000)
target='label'
IDcol = '????'
predictors = [x for x in data_tree_train.columns if x not in [target, IDcol]]

In [ ]:
lgb2 = lgb.LGBMClassifier(
        boosting_type='gbdt', num_leaves=10, max_depth=7, 
        learning_rate=0.1, n_estimators=256, max_bin=255, subsample_for_bin=50000, 
        min_split_gain=0, min_child_weight=1, min_child_samples=10, 
        subsample=1, subsample_freq=1, colsample_bytree=1, reg_alpha=0, reg_lambda=0, 
        seed=0, nthread=-1, silent=True)
lgb2.fit(train[predictors], train['label'])

In [ ]:
feaImp = pd.Series(lgb2.booster().feature_importance()).sort_values(ascending=False)#???????0???
goodFeature = feaImp[feaImp>0].index
features = data_tree_train_0[predictors].columns[goodFeature]
features

In [ ]:
start = time.time()
param_test1 = {
#     'num_leaves':np.array([10,32,100]),
#     'max_depth':np.array([5,10,-1]),
#     'min_child_weight':np.array([1,3,5,10]),
#     'gamma':[i/10.0 for i in range(0,5)],
#     'n_estimators':np.array([100,500,1000])
}
gsearch1 = GridSearchCV(estimator = RandomForestClassifier(n_estimators=200, max_features=0.5), 
                       param_grid = param_test1, scoring='f1',n_jobs=4,iid=False, cv=8)
gsearch1.fit(data_tree_train[predictors],data_tree_train[target])
end = time.time()
print('????:')
(end-start)/60

In [ ]:
gsearch1.grid_scores_, gsearch1.best_params_, gsearch1.best_score_

In [ ]:
start = time.time()
param_test1 = {
#     'num_leaves':np.array([10,32,100]),
#     'max_depth':np.array([5,10,-1]),
#     'min_child_weight':np.array([1,3,5,10]),
#     'gamma':[i/10.0 for i in range(0,5)],
#     'n_estimators':np.array([100,500,1000])
}
gsearch2 = GridSearchCV(estimator = RandomForestClassifier(n_estimators=200, max_features=0.5), 
                       param_grid = param_test1, scoring='f1',n_jobs=4,iid=False, cv=8)
gsearch2.fit(data_tree_train[features],data_tree_train[target])
end = time.time()
print('????:')
(end-start)/60

In [ ]:
gsearch2.grid_scores_, gsearch2.best_params_, gsearch2.best_score_